In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False, # <- 新增
    thinking_budget=1024, # <- 新增，至少要花掉這些 token 進行思考
):
    params = {
        "model": model,
        "max_tokens": 4000, # <- 修改，至少要設定成比 thinking_budget 還要多
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking: # <- 新增
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## 一般 thinking 使用方法:

In [3]:
messages = []

add_user_message(messages, "Write a paragraph guide to recursion")

chat(messages, thinking=True)

Message(id='msg_01T7uFxVHGagt25vvhUbqtmi', container=None, content=[ThinkingBlock(signature='ErAECkYICxgCKkDy3TzsGM0Pb47ZHKoi0iDeMHjqQAPcYcf4kphPH9bMIdWCA2qxevrQACeZ6qaVaWU2vGDEJ2W8e1Zsqs/Ic7tmEgyBPGrRQkrXOpmP0R0aDKd6iWJq26HaORIaAiIwn/noZ9EejrrCdkBjrKTcQgW6mJGJrZEcyQ0nfWE0OCNpIaAxKBLagMFgMuORdoHeKpcDFqQOPgSj9LuWaIQ+0IWAIHMftaBwn658yhh8xjeOY6FvucfPmHjPNekWczNzZ+gw16SJ16gy3iF10hNVJZbEBHzHpPeLXDFPmJhsIinSI6lwC/Nrd5hX+ku9hozckp6VcU50hS92BGmIXhzKBTkjsFWEI955tW41trt1EnVv6ZJiR1qnVLduTxv71wUKymQmDbSmWaeuKqQxEhk3EDImgonI5ee6PBL0uwrW3QktA2cIq09lceYVNfqTyuMYU++XVBtr48SiISgf0ZrXFpTocZqPXY2AzKU9pRsIE/ecwYtgOzJhdrqOV8YM0OEYNn/I4Qy2c+Pe61AfxUDsvaXp6AW+6x5A1sSdc4Fz1eyxV1EhMuI1mIngoR87zf2BC4lZOVkxvlgbMFdHjLINjWXaFCd5UMVmaHFQI+8kstDjvuinpZR40UitsaX8r5LOrR0CeR72PaLN2IW2mZWhoBTAIBI3Gy6vUcnyVCHUagBWLjVtWruD5VMpr/EAMit14DSquhQCgrlOcwYDBMHnzVjk2oJRKY+/RogYAQ==', thinking="The user wants a paragraph guide to recursion. This is a programming concept that can be tricky to understand, so I should explain it clea

## 強制使用 Redacted Thinking
加入

```python
# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"
```

這一串 Magic string ，可以強制讓 Claude 回覆的 thinking process 區塊變成 `redacted_thinking`，後續要處理時比較方便

In [ ]:
messages = []

#這邊是直接拿來使用，當示範而已
add_user_message(messages, thinking_test_str)

chat(messages, thinking=True)

Message(id='msg_0183vZe2HBzE62vptCvwV7sZ', content=[RedactedThinkingBlock(data='EtcGCkYIAxgCKkA5ARFLpCpwMLfr/Yay7pBreer2GmgGShPJOy+XL0GXbOIadqOPUSauVM++DQfK8wqs3t1zTG10GDBFeOiC7OcUEgx5Aqyir9dh1FHGsrQaDICozuiFF40ZkXmIHCIwbz+GLy+Sk5503ESJLS2SCf/MKJ1CrdeF7oti1Khm78tqRvCZD/PP2BB9694uE5PCKr4FDBMr92mHP8jTJe1kb8kFa8Zi14FQGT+2FvN+IyObltyhkvC0NSPhO4Su8LpikN+JdjSsCqEKpqLYS4bPCQQNVpp8cNvPRmmvVpE5sbdF+C1l1hX5L2j95f8PbpBqOSfcIpDOoQ7o6MgxTpHezvXsTZaaXocxz0jea5jaXcCdqBnXnSRJQnxL3EuTnWulniO2+lUBIrek3q493GaX/SQ/2JCHmv/QIYN4wq/R9DXkK+pakUvil3bNB+Sgy5IjR7o/q5XMtmHlgDGuP1l8AxAbyfJJwQIGteBXHfTFFa4Ah+JMk2EIzjGZ6CqCgr5EnoUFkonWVFOBWVyQdM1e8tr3GTNguVX1BlRKmxzQU9/vJZ/CoYSoo8n6GgvCDWWoCTwozWjgjAMtUoltdq2OOq22i+rSHtUBWR/hnQuIsd/BNIFletVW8rgFSyFKJUwPs0CBlr4i31gt7LF6nSvPt+MbB9eZNomBPPO4s7hsdJ5gH2X6SLAleFmccUfiwS8sXp/9QKQvLccOi8xn1gZwADI4uZTihTPWKxv7RzdMYu3uyFwmZb9JZXb4a1H9PSgc7afQWvX2s+A+mUVA1lP4iu0XzGGfekvmP4/MEXKbkY38iqwuuaqnG+UrqCfDWBBzEb/seoDI3W3NnmJLNRMXZlcBjQAYp9F7x9NVt/YVtD9ELtNiaOT6keQeAEcnbGGGwHNZ5Dqdhhnt